# The dial: risk tiers and confidence escalation

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 20 §§20.1–20.3 · type: concept-lab*

**The promise:** by the end you can classify any agent action on the *two* orthogonal axes that send work to a human — **reversibility** (how expensive to undo?) and **confidence** (is the agent probably wrong?) — and route on *either*, with each trigger tuned separately.

## 🧠 Why this matters

Calibrate an agent's autonomy the way you calibrate a new hire's. On day one they propose and you approve everything; as they demonstrate judgment you stop reviewing the routine and keep approval rights over the expensive — anything hard to reverse, customer-visible, or security-sensitive. The whole point of this chapter is to make that dial *explicit in the architecture* rather than implicit in hope.

There are **two independent reasons** to call a human, and confusing them is the classic mistake. The *action* may be costly to undo (reversibility — a property of the tool). Or *this particular answer on this particular input* may be wrong (confidence — a property of the answer). Gating by risk alone sends the confident-but-irreversible case to a human and waves the unsure-but-reversible case straight through — exactly backwards for the second failure. A complete policy is the **union**: escalate when *either* fires.

## Objectives & prerequisites

**By the end you can:**
- Tier any tool by reversibility (read-only → reversible write → hard-to-reverse → irreversible) and read the policy straight off the table.
- Assemble a cheap **confidence score** from two or three noisy signals so their errors don't line up.
- Set a **per-slice deferral threshold** and route a decision to a human when the *risk axis OR the confidence axis* fires.

**Prereqs (concepts, not installs):** Ch 16 (verification feeds a confidence signal; `RunBudget`), Ch 13 (retrieval top-score as a signal), Ch 22 (evals — escalations become labeled data). This notebook is **offline by design**: the lesson is policy and signal-combination, not model calls. Run it top to bottom; no API key needed.

## Setup

Standard companion setup. `MOCK=1` (the default) keeps everything offline and deterministic. One optional cell near the end sketches a *live* self-consistency vote behind `MOCK=0`; the core lesson never needs it.

In [ ]:
import os
import random
from dataclasses import dataclass, field
from enum import Enum

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; secrets come ONLY from the environment

# MOCK=1 (default) => canned, offline, free, deterministic. MOCK=0 => live calls (one optional cell).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

SEED = 20
random.seed(SEED)  # seed anything stochastic so the notebook is reproducible

print(f"MOCK mode: {MOCK}  (set COMPANION_MOCK=0 for the optional live cell)")
print("This notebook runs free and offline in MOCK mode — no API key required.")

## Axis 1 — reversibility: the risk-tier table (§20.2)

What deserves a gate? The same question Ch 27 asks of any decision: *how expensive is this to reverse?* Reading a doc is free to undo (do nothing). Drafting a reply is cheap (discard it). Sending it to a customer is irreversible. Classify every tool on that axis and the policy writes itself.

Crucially, **tiers attach to tools *and* to argument patterns** — a €10 refund and a €10,000 refund are different tiers. So the policy lives in code your security review can read, not in the prompt.

In [ ]:
class Tier(str, Enum):
    READ = "read"              # read-only: autonomous, just log
    REVERSIBLE = "reversible"  # reversible write: autonomous, log + visible undo
    GATED = "gated"            # hard-to-reverse / irreversible: human approval first


# Base tier per tool. Matches the capstone's TOOL_TIERS (§20.6).
TOOL_TIERS = {
    "search_docs":   Tier.READ,
    "get_ticket":    Tier.READ,
    "create_ticket": Tier.REVERSIBLE,
    "send_reply":    Tier.GATED,
    "refund":        Tier.GATED,
}


def tier_for(tool: str, args: dict | None = None) -> Tier:
    """Tier a CALL, not just a tool name: argument patterns can raise the tier.

    Unknown tools default to GATED — fail safe, never fail open.
    """
    base = TOOL_TIERS.get(tool, Tier.GATED)
    args = args or {}
    # A small refund might be reversible-by-policy; a large one is always gated.
    if tool == "refund" and args.get("amount_eur", 0) <= 25:
        return Tier.REVERSIBLE
    return base


POLICY = {
    Tier.READ:       "Autonomous; log.",
    Tier.REVERSIBLE: "Autonomous; log + visible undo.",
    Tier.GATED:      "Human approval before execution.",
}

for tool in TOOL_TIERS:
    t = tier_for(tool)
    print(f"{tool:<14} {t.value:<11} -> {POLICY[t]}")

Notice the same tool can land in different tiers depending on its arguments — that is the "€10 vs €10,000 refund" rule made concrete:

In [ ]:
small = tier_for("refund", {"amount_eur": 10})
large = tier_for("refund", {"amount_eur": 4900})
print(f"refund €10    -> {small.value}")
print(f"refund €4900  -> {large.value}")
assert small is Tier.REVERSIBLE and large is Tier.GATED

### ⚠️ Pitfall: approval fatigue

Approval fatigue is the failure mode that eats this whole chapter. Gate *too much* and within a week humans click "approve" reflexively — you now have the latency of oversight with none of the safety, and your audit log "proves" a human reviewed the disaster nobody read.

Gates are a **scarce attention budget**. A gate that approves 99.8% of what it sees is not a control — it is a candidate for *automation with sampled audits* (Ch 23). So: measure approval rates and retire rubber-stamp gates. Let's make that measurable.

In [ ]:
def approval_rate(decisions: list[bool]) -> float:
    """Fraction of gated requests a human approved. ~1.0 == rubber-stamp."""
    return sum(decisions) / len(decisions) if decisions else 0.0


# A week of human decisions on one gate (True == approved).
gate_history = [True] * 499 + [False] * 1  # 499 approvals, 1 denial
rate = approval_rate(gate_history)
print(f"approval rate: {rate:.3%}")
if rate > 0.98:
    print("⚠️  Rubber-stamp gate: convert to automation + sampled audits, not a human click.")

## Axis 2 — confidence: *is the agent probably wrong?* (§20.3)

Reversibility is blind to a second, orthogonal question. You can be near-certain about an irreversible action (the refund is correct — gate it once, then let it run) and badly unsure about a trivially reversible one (a draft you'd ship to nobody without a second look).

Escalation needs a number — *how sure is the agent?* — and no single signal gives a trustworthy one. Each is a noisy proxy; the engineering is **combining a few cheap ones so their errors don't line up**. The chapter's signal menu:

In [ ]:
@dataclass
class Signals:
    """Noisy confidence proxies (§20.3). All scaled so HIGHER == MORE confident."""
    retrieval_top_score: float = 1.0   # Ch 13: low top score predicts hallucination
    self_report_unsure: float = 0.0    # Ch 10/13: model's own "I'm not sure" flag in [0,1]
    consistency_agreement: float = 1.0 # Ch 10: agreement across N self-consistency samples


def confidence(signals: Signals) -> float:
    """Combine cheap signals conservatively: the WEAKEST link sets confidence.

    Taking the min means any one signal screaming 'unsure' pulls the whole score
    down — errors have to ALL be optimistic at once to slip through.
    """
    return min(
        signals.retrieval_top_score,        # low retrieval => likely wrong
        1.0 - signals.self_report_unsure,   # model's own escape hatch
        signals.consistency_agreement,      # N-sample agreement
    )


print("confident :", round(confidence(Signals(0.91, 0.05, 0.95)), 3))
print("shaky     :", round(confidence(Signals(0.42, 0.60, 0.50)), 3))

## The decision: `should_escalate` routes on *either* axis (§20.3)

The two triggers compose cleanly *because they are independent*. One line checks the tool's reversibility tier; the rest score the answer's confidence; **either** routes to the same human queue — for different reasons, captured with different metadata, tuned by different knobs.

In [ ]:
@dataclass
class Decision:
    tool: str
    args: dict = field(default_factory=dict)
    signals: Signals = field(default_factory=Signals)
    slice_threshold: float = 0.7  # per-slice; tune separately per intent (see below)


def should_escalate(d: Decision) -> tuple[bool, str]:
    """Escalate when the action is risky OR the agent is unsure. Returns (escalate?, why)."""
    if tier_for(d.tool, d.args) is Tier.GATED:
        return True, "risk: gated tool/args (costly to reverse)"   # axis 1
    conf = confidence(d.signals)
    if conf < d.slice_threshold:                                   # axis 2
        return True, f"confidence {conf:.2f} < threshold {d.slice_threshold:.2f}"
    return False, f"auto: tier ok, confidence {conf:.2f} >= {d.slice_threshold:.2f}"

Here is the fixture — a small, deterministic, in-cell list of mock decisions (tool + args + signal values). Read each one and form an opinion before you run the router.

In [ ]:
MOCK_DECISIONS = [
    # confident read-only lookup -> should auto-run
    Decision("search_docs", {"q": "refund policy"}, Signals(0.93, 0.02, 0.98)),
    # reversible write, well-grounded -> should auto-run
    Decision("create_ticket", {"subject": "export help"}, Signals(0.88, 0.05, 0.90)),
    # GATED tool, even though the model is confident -> escalate on RISK
    Decision("send_reply", {"to": "ana@example.com"}, Signals(0.95, 0.01, 0.97)),
    # small refund => reversible tier, but signals are SHAKY -> escalate on CONFIDENCE
    Decision("refund", {"amount_eur": 10}, Signals(0.35, 0.70, 0.40)),
    # unknown tool defaults to GATED -> escalate on RISK
    Decision("wire_transfer", {"amount_eur": 5000}, Signals(0.99, 0.0, 0.99)),
]

### 🔮 Predict

Before running the next cell, decide for each of the five decisions: **escalate or auto-run?** — and *which axis* triggers it (risk vs confidence). Two are auto-runs; three escalate for three different reasons. Write your guesses down, then run.

In [ ]:
for d in MOCK_DECISIONS:
    esc, why = should_escalate(d)
    label = "🛑 ESCALATE" if esc else "✅ auto-run "
    print(f"{label}  {d.tool:<14} {why}")

**What you just saw.** `send_reply` escalated despite a 0.95 retrieval score — risk, not confidence, tripped it. The €10 `refund` is a *reversible* tier, yet it escalated because its signals were shaky — confidence, not risk. And `wire_transfer`, a tool the table never heard of, defaulted to gated. Two axes, one queue, three different reasons. *Gating by risk alone would have waved that shaky refund straight through.*

## The threshold is a knob, set per slice (§20.3)

The threshold is **not a constant to discover** — it is a knob you set, trading one cost against another. Too high and you *over-defer*: automation collapses and reviewers drown until they stop reading (approval fatigue, through a different door). Too low and you *under-defer*: confident errors ship that no human ever saw.

Tune it against two numbers — the **cost of a wrong answer** and the **cost of human time** — and make it **per slice**: stricter on the expensive intents (legal, financial, safety), looser on the routine. Watch the same shaky decision route differently under a lenient vs a strict slice:

In [ ]:
shaky = Signals(0.62, 0.20, 0.66)  # confidence == min(0.62, 0.80, 0.66) == 0.62

SLICE_THRESHOLDS = {
    "faq_lookup":   0.50,  # cheap, reversible mistakes -> defer rarely
    "billing":      0.75,  # money on the line -> defer eagerly
    "legal_answer": 0.90,  # expensive to be wrong -> defer very eagerly
}

for slice_name, thr in SLICE_THRESHOLDS.items():
    d = Decision("create_ticket", {}, shaky, slice_threshold=thr)  # reversible tool
    esc, why = should_escalate(d)
    print(f"{slice_name:<13} thr={thr:.2f} -> {'ESCALATE' if esc else 'auto-run'}  ({why})")

Same answer, same confidence (0.62). Auto-runs as an FAQ lookup; escalates under billing and legal. The slice *is* the policy.

## (Optional, live) Self-consistency as a confidence signal — `MOCK=0`

Self-consistency is the most useful *expensive* signal: sample the same question N times and measure how much the answers agree. In `MOCK=1` we fake the votes deterministically. Set `COMPANION_MOCK=0` and provide `ANTHROPIC_API_KEY` to run it for real (rough cost: N short completions, a few hundred tokens — cents). The signal-combination lesson does **not** require this cell.

In [ ]:
from collections import Counter


def self_consistency_agreement(question: str, n: int = 5) -> float:
    """Sample an answer n times; return the fraction agreeing with the plurality."""
    if MOCK:
        # Canned, seeded votes: 4 of 5 agree -> agreement 0.8. Deterministic.
        rng = random.Random(SEED)
        votes = rng.sample(["yes", "yes", "yes", "yes", "no"], k=5)
    else:
        # Live path: the latest, most capable Claude model, temperature>0 for diversity.
        from anthropic import Anthropic  # imported lazily so MOCK runs need no SDK key

        client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
        votes = []
        for _ in range(n):
            resp = client.messages.create(
                model="claude-opus-4-5",
                max_tokens=5,
                temperature=1.0,
                messages=[{"role": "user", "content": f"{question} Answer yes or no."}],
            )
            votes.append(resp.content[0].text.strip().lower()[:3])
    top, count = Counter(votes).most_common(1)[0]
    return count / len(votes)


agree = self_consistency_agreement("Is a duplicate-charge refund covered by policy?")
print(f"self-consistency agreement: {agree:.2f}  (feeds Signals.consistency_agreement)")

## 🎯 Senior lens

A senior treats every escalation as **free training data**. An input the agent found hard, paired with the human's correct resolution, is exactly the labeled hard case your eval set is starving for (Ch 22). So the deferral threshold is not a static safety valve — it is a *sampling strategy* that mines your weakest slices. Wire the escalation queue into the dataset flywheel and the evidence accumulates to justify moving the threshold **down** over time: the dial of §20.1, now applied to confidence and turned by data instead of nerve.

## Recap

- **Two orthogonal axes** send work to a human: reversibility (a property of the *action*) and confidence (a property of *this answer*). A complete policy is their **union** — escalate when *either* fires.
- **Tier tools in code**, including argument patterns (€10 vs €10,000), and default unknown tools to **GATED** — fail safe.
- **Confidence = a combination of cheap, noisy signals** (`min` of retrieval score, the inverse self-report flag, self-consistency) so their errors don't line up; reserve expensive signals for costly slices.
- The **deferral threshold is a per-slice knob**, tuned against cost-of-wrong-answer and cost-of-human-time — not a constant.
- **Approval fatigue** is the meta-failure: measure approval rates and retire rubber-stamp gates (>98% approve) into sampled audits.
- Every escalation is a **labeled hard case** — pipe it to the Ch 22 flywheel and move the dial with evidence.

## Exercises

Each task *changes* something and asks you to predict the result first. Solutions live in `solutions/` (Phase 2), not inline.

1. **Add a per-argument tier rule.** Make `send_reply` only GATED when the recipient is external (not an `@yourcompany.com` address); internal replies become REVERSIBLE. Predict which of `MOCK_DECISIONS` change, then verify.
2. **Swap the combiner.** Replace `min` in `confidence` with a weighted average (e.g. retrieval 0.5, self-report 0.3, consistency 0.2). Predict how the shaky `refund` decision moves, then check whether a strong signal now masks a weak one — and argue which combiner is safer.
3. **Tune a slice.** Pick the `billing` slice and find the threshold at which the €10 `refund` flips from escalate to auto-run. What does that threshold imply about your cost-of-wrong-answer?
4. **Measure fatigue over time.** Simulate 4 weeks of gate decisions where the approval rate climbs from 92% to 99.9%. Write the rule that *automatically* flags the gate for retirement, and decide what sampling rate the replacement audit should use.

In [ ]:
# Exercise 1 — internal vs external recipient changes the tier.


In [ ]:
# Exercise 2 — weighted-average combiner vs min(); does a strong signal mask a weak one?


In [ ]:
# Exercise 3 — find the billing-slice threshold where the €10 refund flips.


In [ ]:
# Exercise 4 — simulate climbing approval rates; rule to retire a rubber-stamp gate.


## Next

You can now route a decision to a human on *either* axis — but routing is only half the mechanism. **Next:** [`20-02-approval-gates-in-the-loop.ipynb`](./20-02-approval-gates-in-the-loop.ipynb) turns the GATED verdict into a real **park-as-`waiting_human` → approve/deny → resume** cycle inside the agent loop (the chapter's 🔧 Build, §20.6).

- **Blueprint:** the tier-aware executor hardens [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/) and gates the team in [`../../../blueprints/multi-agent-supervisor/`](../../../blueprints/multi-agent-supervisor/).
- **Capstone:** this feeds `capstone/approvals.py` and the gate in `capstone/loop.py` — checkpoint `checkpoints/ch20-approval-gates`.